In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    f1_score
)



Predict whether a player scores a goal during a shift, using only information available at the beginning of that shift.

- **Target:** `player_goal` — 1 if the player scores during the shift, otherwise 0.
- **Training set:** Used to fit the models.
- **Validation set:** Used to compare models and select a prediction threshold.
- **Test set:** Used once for final evaluation.

Because goals during an individual shift are relatively rare, the main evaluation metric is **average precision (AP)**, alongside ROC-AUC, Brier score, log loss, precision, and recall.


In [2]:
input_df = pd.read_csv("Data/model_input_data.csv")

feature_cols = [
    "points_l5_games",
    "goals_l5_games",
    "shots_l5_games",
    "oz_faceoff",
    "score_diff",
    "giveaways_l5",
    "Goalie_Pulled",
    "takeaways_l5",
    "shot-on-goal_l5",
    "delayed-penalty",
    "PP",
    "cumsum_icetime",
    "oponent_goalAgainst",
    "oponent_pointPctg",
    "m_left_in_game",
    "home_game"
]

target_col = "player_goal"

input_df['m_left_in_game'] = input_df['startTime'].apply(lambda x: 60 - (int(x[0])*10 +  int(x[1]) + int(x[4])/6 +  int(x[4])/60))


required_cols = feature_cols + [target_col]
missing_cols = [c for c in required_cols if c not in input_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Drop rows without a known target.
df = input_df.dropna(subset=[target_col]).copy()

# Make sure the target is binary.
if not df[target_col].isin([0, 1]).all():
    raise ValueError("player_goal must contain only 0 and 1.")

df[target_col] = df[target_col].astype(int)

# Convert features to numeric values.
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Rows:", len(df))
print("Goal shifts:", int(df[target_col].sum()))
print("Goal-shift rate:", round(df[target_col].mean(), 4))

print("\nMissing values:")
print(df[feature_cols].isna().sum().sort_values(ascending=False))

print("\nFeature ranges:")
print(df[feature_cols].describe().T)


Rows: 4864
Goal shifts: 215
Goal-shift rate: 0.0442

Missing values:
oponent_goalAgainst    68
oponent_pointPctg      68
points_l5_games        24
goals_l5_games         24
shots_l5_games         24
oz_faceoff              0
score_diff              0
giveaways_l5            0
Goalie_Pulled           0
takeaways_l5            0
shot-on-goal_l5         0
delayed-penalty         0
PP                      0
cumsum_icetime          0
m_left_in_game          0
home_game               0
dtype: int64

Feature ranges:
                      count       mean       std    min        25%        50%  \
points_l5_games      4840.0   8.044628  2.857588   1.00   6.000000   8.000000   
goals_l5_games       4840.0   2.300413  1.612439   0.00   1.000000   2.000000   
shots_l5_games       4840.0  16.955579  5.377916   2.00  13.000000  17.000000   
oz_faceoff           4864.0   0.264186  1.077266  -4.00   0.000000   0.000000   
score_diff           4864.0   0.249178  1.742998  -8.00  -1.000000   0.000000   

In [3]:
date_col = "gameDate"

if date_col not in df.columns:
    raise ValueError(
        "Add a game-date column before splitting chronologically."
    )

df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=[date_col]).copy()

# Sort before splitting.
df = df.sort_values([date_col]).reset_index(drop=True)

# Keep every game date entirely within one split.
dates = np.array(sorted(df[date_col].unique()))

if len(dates) < 3:
    raise ValueError("Not enough distinct dates for three chronological splits.")

train_end = int(len(dates) * 0.70)
val_end = int(len(dates) * 0.85)

train_dates = dates[:train_end]
val_dates = dates[train_end:val_end]
test_dates = dates[val_end:]

train_df = df[df[date_col].isin(train_dates)].copy()
val_df = df[df[date_col].isin(val_dates)].copy()
test_df = df[df[date_col].isin(test_dates)].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nPositive target rates:")
print("Train:", round(y_train.mean(), 4))
print("Validation:", round(y_val.mean(), 4))
print("Test:", round(y_test.mean(), 4))

Training rows: 3345
Validation rows: 723
Test rows: 796

Positive target rates:
Train: 0.0377
Validation: 0.0719
Test: 0.0465


In [4]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight=None,
        random_state=13
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=13
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=20,
        random_state=13
    )
}

In [5]:
from sklearn.preprocessing import StandardScaler

def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", clone(model))
    ])

fitted_models = {}
validation_probabilities = {}
test_probabilities = {}
results = []

for name, model in models.items():
    pipeline = make_pipeline(model)
    pipeline.fit(X_train, y_train)

    val_prob = pipeline.predict_proba(X_val)[:, 1]
    test_prob = pipeline.predict_proba(X_test)[:, 1]

    fitted_models[name] = pipeline
    validation_probabilities[name] = val_prob
    test_probabilities[name] = test_prob

    results.append({
        "model": name,
        "validation_AP": average_precision_score(y_val, val_prob),
        "test_AP": average_precision_score(y_test, test_prob),
        "test_ROC_AUC": roc_auc_score(y_test, test_prob),
        "test_Brier": brier_score_loss(y_test, test_prob),
        "test_LogLoss": log_loss(y_test, test_prob, labels=[0, 1])
    })

results_df = pd.DataFrame(results).sort_values(
    "validation_AP", ascending=False
)

print(results_df.round(4).to_string(index=False))


              model  validation_AP  test_AP  test_ROC_AUC  test_Brier  test_LogLoss
  Gradient Boosting         0.3606   0.2458        0.8121      0.0411        0.1614
      Random Forest         0.3509   0.2495        0.7831      0.0426        0.1721
Logistic Regression         0.2514   0.0723        0.6777      0.0448        0.1842


In [6]:
candidate_thresholds = np.arange(0.01, 0.51, 0.01)

chosen_thresholds = {}

for name, probs in validation_probabilities.items():
    threshold_scores = []

    for threshold in candidate_thresholds:
        pred = (probs >= threshold).astype(int)

        threshold_scores.append({
            "threshold": threshold,
            "precision": precision_score(
                y_val, pred, zero_division=0
            ),
            "recall": recall_score(
                y_val, pred, zero_division=0
            ),
            "f1": f1_score(
                y_val, pred, zero_division=0
            ),
            "flagged_shifts": int(pred.sum())
        })

    threshold_df = pd.DataFrame(threshold_scores)

    # Select using validation results only.
    best = threshold_df.loc[threshold_df["f1"].idxmax()]
    chosen_thresholds[name] = float(best["threshold"])

    print(f"\n{name}")
    print(best.round(3).to_string())


Logistic Regression
threshold          0.110
precision          0.304
recall             0.269
f1                 0.286
flagged_shifts    46.000

Random Forest
threshold          0.230
precision          0.290
recall             0.385
f1                 0.331
flagged_shifts    69.000

Gradient Boosting
threshold          0.240
precision          0.857
recall             0.231
f1                 0.364
flagged_shifts    14.000


In [7]:
final_results = []

for name in models:
    probs = test_probabilities[name]
    threshold = chosen_thresholds[name]
    pred = (probs >= threshold).astype(int)

    final_results.append({
        "model": name,
        "threshold": threshold,
        "test_AP": average_precision_score(y_test, probs),
        "test_ROC_AUC": roc_auc_score(y_test, probs),
        "test_Brier": brier_score_loss(y_test, probs),
        "test_LogLoss": log_loss(y_test, probs, labels=[0, 1]),
        "flagged_shifts": int(pred.sum()),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

final_results_df = pd.DataFrame(final_results).sort_values(
    "test_AP", ascending=False
)

print(final_results_df.round(4).to_string(index=False))


              model  threshold  test_AP  test_ROC_AUC  test_Brier  test_LogLoss  flagged_shifts  precision  recall     F1
      Random Forest       0.23   0.2495        0.7831      0.0426        0.1721              61     0.1967  0.3243 0.2449
  Gradient Boosting       0.24   0.2458        0.8121      0.0411        0.1614               5     0.6000  0.0811 0.1429
Logistic Regression       0.11   0.0723        0.6777      0.0448        0.1842              40     0.0750  0.0811 0.0779



- **Precision:** Of the shifts flagged by the model, what fraction actually contained a goal?
- **Recall:** Of all goal-scoring shifts, what fraction did the model flag?
- **Average precision:** How well does the model rank goal-scoring shifts above non-scoring shifts across different thresholds?
- **ROC-AUC:** How well does the model rank positive cases above negative cases?
- **Brier score and log loss:** How good are the predicted probabilities? Lower is better.


In [8]:
best_model_name = "Gradient Boosting"

artifact = {
    "model": fitted_models[best_model_name],
    "threshold": chosen_thresholds[best_model_name],
    "features": feature_cols,
    "model_name": best_model_name
}

import joblib
joblib.dump(artifact, "nhl_shift_goal_model.joblib")

print("Saved:", "nhl_shift_goal_model.joblib")



Saved: nhl_shift_goal_model.joblib


## Conclusion

This model compares three approaches to predicting whether a player scores during an individual shift. The rare-event nature of the target makes precision-recall evaluation more informative than accuracy alone. A useful model should rank goal-scoring shifts highly and provide a practical balance between precision and recall.

The next priority is validating the data-generation process: ensure every feature is available at shift start, correct the game-clock calculation, and verify suspicious negative event counts. After that, use chronological validation to select a model and threshold, and reserve the final test period for an unbiased performance estimate.

The current results are promising, but they are preliminary. More games and evaluation across additional future periods will help determine whether the performance generalizes.
